In [ ]:
import os 
import numpy as np
import scipy.io as sio
from scipy.signal import butter, filtfilt

def apply_strict_bandpass(data, lowcut=8.0, highcut=14.0, fs=250.0, order=4):
    # Tightened to 8-14Hz to target the Mu-rhythm specifically
    nyq = 0.5 * fs
    low, high = lowcut / nyq, highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, data, axis=0)

def apply_voltage_gate(epoch, threshold=2.5):
    # Clips outliers to prevent eye blinks from dominating the features
    return np.clip(epoch, -threshold, threshold)

def process_subject_v4(mat_file_path, subject_id):
    print(f"Loading {mat_file_path} for strict processing...")
    mat_data = sio.loadmat(mat_file_path)
    all_runs = mat_data['data'][0]

    SFREQ = 250
    Action_start, Action_end = int(0.5*SFREQ), int(2.5*SFREQ)
    Baseline_start, baseline_end = int(-0.2*SFREQ), 0

    all_clean_epochs, all_clean_labels = [], []

    for run_idx, run in enumerate(all_runs):
        X, y, trial = run['X'].item(), run['y'].item(), run['trial'].item()
        if len(y) == 0: continue

        # 1. Filter continuous data (Alpha band only)
        filtered_eeg = apply_strict_bandpass(X[:, :22])

        for i in range(len(trial)):
            start_idx = trial[i][0]
            epoch = filtered_eeg[start_idx + Action_start : start_idx + Action_end, :]
            baseline = filtered_eeg[start_idx + Baseline_start : start_idx + baseline_end, :]

            # 2. Calculate local stats for normalization
            b_mean = np.mean(baseline, axis=0)
            b_std = np.std(baseline, axis=0)
            
            # 3. Normalize and Apply the Strict Voltage Gate
            norm_epoch = (epoch - b_mean) / (b_std + 1e-6)
            gated_epoch = apply_voltage_gate(norm_epoch, threshold=2.5)
            
            # 4. Global Artifact Rejection (Throw away entirely if still too messy)
            if np.max(np.abs(gated_epoch)) < 10: 
                all_clean_epochs.append(gated_epoch)
                all_clean_labels.append(y[i][0] - 1)
        
        print(f"Run {run_idx+1} complete.")

    final_X = np.array(all_clean_epochs).transpose(0, 2, 1)
    final_y = np.array(all_clean_labels)

    os.makedirs("processed_data", exist_ok=True)
    np.save(f"processed_data/sub_{subject_id}_X_v4.npy", final_X)
    np.save(f"processed_data/sub_{subject_id}_y_v4.npy", final_y)
    print(f"Saved {len(final_y)} clean, gated trials.")

process_subject_v4('TrainingTesting-Data/A09T.mat', '09')

📂 Loading TrainingTesting-Data/A09T.mat for strict processing...
✅ Run 4 complete.
✅ Run 5 complete.
✅ Run 6 complete.
✅ Run 7 complete.
✅ Run 8 complete.
✅ Run 9 complete.
🎉 Saved 288 clean, gated trials.


In [ ]:
def apply_strict_gate(epoch, threshold=3.0):
    """
    If any electrode in a trial exceeds 3 standard deviations, 
    we clip it to prevent the AI from 'learning' eye blinks.
    """
    return np.clip(epoch, -threshold, threshold)

# In your processing loop, add:
normalized_epoch = (epoch - b_mean) / (b_std + 1e-6)
# NEW: Strict clipping gate
normalized_epoch = apply_strict_gate(normalized_epoch, threshold=2.5)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
import numpy as np

# --- 1. LOADING & SPLITTING BOTH X AND Y ---
X = np.load('processed_data/sub_09_X.npy')
y = np.load('processed_data/sub_09_y.npy')

# The "y" is essential here; we stratify to keep class balance even
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Convert to Tensors
train_loader = DataLoader(TensorDataset(torch.from_numpy(X_train).float(), torch.from_numpy(y_train).long()), batch_size=32, shuffle=True)
test_loader = DataLoader(TensorDataset(torch.from_numpy(X_test).float(), torch.from_numpy(y_test).long()), batch_size=32, shuffle=False)

import torch
import torch.nn as nn
import numpy as np

class MinimalistSpatialExtractor(nn.Module):
    def __init__(self):
        super(MinimalistSpatialExtractor, self).__init__()
        # 4 Virtual Channels. No training = No memorization.
        self.spatial_gate = nn.Conv1d(22, 4, kernel_size=1)
        
    def forward(self, x):
        # AI PATH: Simple Spatial Energy
        virt_channels = self.spatial_gate(x)
        ai_feat = torch.mean(torch.abs(virt_channels), dim=2) 
        
        # PHYSICS PATH: Your trusted stats
        var_f = torch.var(x, dim=2)
        rms_f = torch.sqrt(torch.mean(x**2, dim=2))
        ptp_f = torch.max(x, dim=2)[0] - torch.min(x, dim=2)[0]
        
        # 4 (AI) + 66 (Physics) = 70 Total Features
        fused = torch.cat((ai_feat, var_f, rms_f, ptp_f), dim=1)
        return fused

# --- 1. Load the v4 Gated Data ---
X_v4 = np.load('processed_data/sub_09_X_v4.npy')
y_v4 = np.load('processed_data/sub_09_y_v4.npy')

# --- 2. Extract Features (WITHOUT TRAINING) ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MinimalistSpatialExtractor().to(device)
model.eval() # Set to evaluation mode

def get_static_features(data):
    # Convert entire dataset to tensor
    tensor_x = torch.from_numpy(data).float().to(device)
    with torch.no_grad(): # This prevents the RuntimeError!
        fused_features = model(tensor_x)
    return fused_features.cpu().numpy()

X_features = get_static_features(X_v4)

# --- 3. Split and Train the Forest ---
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(
    X_features, y_v4, test_size=0.2, random_state=42, stratify=y_v4
)

# Using the "Wide and Shallow" configuration to ensure robustness
rf = RandomForestClassifier(
    n_estimators=1000, 
    max_depth=6, 
    max_features='sqrt', 
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)

rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

print(f"Minimalist Hybrid Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")

🔥 Minimalist Hybrid Accuracy: 24.14%


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim

# --- 1. LOSS DEFINITION ---
# CrossEntropyLoss is the gold standard for multi-class intent (Left, Right, Feet, Tongue)
criterion = nn.CrossEntropyLoss()

# --- 2. THE TRAINING LOOP (WITH LOSS) ---
print("--- TRAINING EXTRACTOR WITH CROSS-ENTROPY LOSS ---")
model.train()

for epoch in range(60): # Increased slightly to ensure convergence
    running_loss = 0.0
    correct = 0
    total = 0
    
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        
        optimizer.zero_grad()
        
        # Forward Pass
        preds, _ = model(bx)
        
        # CALCULATE THE ERROR (The Loss Function)
        loss = criterion(preds, by)
        
        # BACKPROPAGATION (The "Learning" Step)
        loss.backward()
        optimizer.step()
        
        # Track progress
        running_loss += loss.item()
        _, predicted = torch.max(preds.data, 1)
        total += by.size(0)
        correct += (predicted == by).sum().item()
        
    if (epoch + 1) % 15 == 0:
        print(f"Epoch [{epoch+1}/60] | Loss: {running_loss/len(train_loader):.4f} | Accuracy: {100*correct/total:.2f}%")

print("Training Complete. Now re-extracting features with optimized weights...")
# [Insert the get_feats and save logic from the previous step here to refresh the .npy files]

--- TRAINING EXTRACTOR WITH CROSS-ENTROPY LOSS ---
Epoch [15/60] | Loss: 0.5787 | Accuracy: 77.39%
Epoch [30/60] | Loss: 0.5040 | Accuracy: 81.74%
Epoch [45/60] | Loss: 0.4808 | Accuracy: 80.87%
Epoch [60/60] | Loss: 0.5700 | Accuracy: 82.17%
Training Complete. Now re-extracting features with optimized weights...


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# 1. Load the refreshed 98-D features
X_tr = np.load('processed_data/X_train_98.npy')
X_te = np.load('processed_data/X_test_98.npy')
y_tr = np.load('processed_data/y_train_98.npy')
y_te = np.load('processed_data/y_test_98.npy')

# 2. The Forest Architecture
# Depth=5 prevents memorizing noise; n_estimators=500 gives massive breadth
rf = RandomForestClassifier(
    n_estimators=500, 
    max_depth=5, 
    max_features='sqrt', 
    class_weight='balanced_subsample',
    n_jobs=-1,
    random_state=42
)

# 3. Train and Evaluate
rf.fit(X_tr, y_tr)
y_pred = rf.predict(X_te)

print(f"\nFinal Hybrid Pipeline Accuracy: {accuracy_score(y_te, y_pred) * 100:.2f}%")
print("\nConfusion Matrix:")
print(confusion_matrix(y_te, y_pred))
print("\nDetailed Report:")
print(classification_report(y_te, y_pred, target_names=["Left", "Right", "Feet", "Tongue"]))


🔥 Final Hybrid Pipeline Accuracy: 25.86%

Confusion Matrix:
[[5 5 3 2]
 [5 3 0 6]
 [4 5 3 3]
 [3 2 5 4]]

Detailed Report:
              precision    recall  f1-score   support

        Left       0.29      0.33      0.31        15
       Right       0.20      0.21      0.21        14
        Feet       0.27      0.20      0.23        15
      Tongue       0.27      0.29      0.28        14

    accuracy                           0.26        58
   macro avg       0.26      0.26      0.26        58
weighted avg       0.26      0.26      0.26        58



In [12]:
import matplotlib.pyplot as plt

# 1. Train the forest again (same code as your last run)
rf.fit(X_tr, y_tr)

# 2. Audit the features
importances = rf.feature_importances_
std = np.std([tree.feature_importances_ for tree in rf.estimators_], axis=0)

# Visualize the top 20 most important features
indices = np.argsort(importances)[-20:]

print("--- TOP 10 MOST INFLUENTIAL FEATURES ---")
feature_names = [f"AI_{i}" for i in range(32)] + [f"Var_{i}" for i in range(22)] + [f"RMS_{i}" for i in range(22)] + [f"PTP_{i}" for i in range(22)]

for i in reversed(indices[-10:]):
    print(f"{feature_names[i]}: {importances[i]:.4f}")

--- TOP 10 MOST INFLUENTIAL FEATURES ---
AI_11: 0.0746
AI_2: 0.0703
AI_3: 0.0703
AI_10: 0.0700
AI_13: 0.0599
AI_6: 0.0552
AI_7: 0.0532
AI_5: 0.0526
AI_9: 0.0513
AI_15: 0.0454


In [7]:
import numpy as np

def extract_pure_physics(X_data):
    # X_data shape: (Trials, 22, 500)
    
    # 1. Power/Variance (ERD/ERS)
    var_f = np.var(X_data, axis=2)
    # 2. Energy (RMS)
    rms_f = np.sqrt(np.mean(X_data**2, axis=2))
    # 3. Peak-to-Peak (Amplitude range)
    ptp_f = np.ptp(X_data, axis=2)
    # 4. Complexity (Standard Deviation)
    std_f = np.std(X_data, axis=2)
    
    # Stack them: (Trials, 22*4 = 88 features)
    features = np.hstack([var_f, rms_f, ptp_f, std_f])
    return features

# Load the filtered data from your .mat processing script
X_full = np.load('processed_data/sub_09_X.npy')
y_full = np.load('processed_data/sub_09_y.npy')

# Extract features
X_phys = extract_pure_physics(X_full)

# Split 80/20
from sklearn.model_selection import train_test_split
X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X_phys, y_full, test_size=0.2, random_state=42, stratify=y_full
)

print(f"Physics-Only Feature Shape: {X_train_p.shape}") # Should be (Trials, 88)

Physics-Only Feature Shape: (230, 88)


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

rf_baseline = RandomForestClassifier(
    n_estimators=1000, # More trees since features are simpler
    max_depth=7,       # Slightly deeper since these features are more stable
    max_features='sqrt',
    class_weight='balanced',
    random_state=42
)

rf_baseline.fit(X_train_p, y_train_p)
y_pred_p = rf_baseline.predict(X_test_p)

print(f"Physics-Only Accuracy: {accuracy_score(y_test_p, y_pred_p) * 100:.2f}%")

✅ Physics-Only Accuracy: 20.69%
